In [1]:
import sys
sys.path.append("..")


In [2]:

import shared_libraries.data_processing_utils as processing
import pandas as pd
import pickle

from typing import Dict, List
from fastf1.core import Session

In [3]:
with open("../ig_sessions.pickle", "rb") as file:
    sessions: List[Session] = pickle.load(file)

In [4]:
compounds_map = pd.read_csv("../ig_compounds_map.csv")

In [62]:
# Create queue of compound data
compd_q = []
for idx in compounds_map.index:
    compd_q.append(compounds_map.loc[idx, :])
compd_q.reverse()

# Merge data from queue with sessions
sess_n_compds = []
for s in sessions:
    s_info = s.session_info
    while True:
        compds = compd_q.pop()
        year_matches = compds["year"] == s_info["StartDate"].year
        name_matches = compds["gp"] == s_info["Meeting"]["Name"]
        if year_matches and name_matches:
            break
    sess_n_compds.append((s, compds))

In [63]:
sess_compds_by_circuit = {}
for s, cpds in sess_n_compds:
    circuit = s.session_info["Meeting"]["Circuit"]["ShortName"]
    if circuit not in sess_compds_by_circuit.keys():
        sess_compds_by_circuit[circuit] = []
    sess_compds_by_circuit[circuit].append((s, cpds))


In [64]:
# Create raw lap data, still split by session
raw_data_by_circuit_split: Dict[str, List[pd.DataFrame]] = {}
for c, ss_n_cs in sess_compds_by_circuit.items():
    raw_data_by_circuit_split[c] = []
    for s, cmpds in ss_n_cs:
        # Get compound mapping
        mapping = cmpds.loc[["soft", "medium", "hard"]]
        # Extract lap data from sessions
        lap_data = processing.get_lap_data_with_weather(s)
        fitted_mapping = pd.concat([pd.DataFrame(mapping).T] * lap_data.shape[0], ignore_index=True)
        # Join lap data with compound data
        lap_data_with_cpds = pd.concat([lap_data, fitted_mapping], axis="columns")
        raw_data_by_circuit_split[c].append(lap_data_with_cpds)


In [65]:

for c, dfs in raw_data_by_circuit_split.items():
    for df in dfs:
        # Add lap time z-score for each lap, calculated locally for each session and 
        processing.add_z_score_for_laps(df)
        # Add IsPitLap column de
        processing.add_is_pit_lap(df)

In [66]:
# Concatenate laps for each circuit
raw_data_by_circuit: Dict[str, pd.DataFrame] = {}
for c, dfs in raw_data_by_circuit_split.items():
    raw_data_by_circuit[c] = pd.concat(dfs, axis="index").reset_index()

In [67]:
# Create a "RealCompund" column which contains the actual compound (C1, C2, ..., or C5) used
for df in raw_data_by_circuit.values():
    df["RealCompound"] = None
    for idx in df.index:
        if df.loc[idx, "Compound"] in ("HARD", "MEDIUM", "SOFT"):
            df.loc[idx, "RealCompound"] = df.loc[idx, df.loc[idx, "Compound"].lower()] # type: ignore
        else:
            df.loc[idx, "RealCompound"] = df.loc[idx, "Compound"]
        



In [112]:
selected_columns = [
    "LapTimeZScore",
    "IsPitLap",
    "Compound",
    "RealCompound",
    "TyreLife",
    "FreshTyre",
    "LapNumber",
    "AirTemp",
    "Humidity",
    "Pressure",
    "Rainfall",
    "TrackTemp",
    "WindDirection"
]
# Clean data for each circuit
clean_data_by_circuit: Dict[str, pd.DataFrame] = {}
for c, df in raw_data_by_circuit.items():
    cleaned = df.loc[:, selected_columns].convert_dtypes().dropna()
    clean_data_by_circuit[c] = cleaned

In [113]:
data = clean_data_by_circuit.copy()

In [114]:
allowed_compounds = ["SOFT", "MEDIUM", "HARD"]
for c, df in data.items():
    data[c] = df.loc[df["Compound"].isin(allowed_compounds), :].copy() # type: ignore

In [ ]:
# Pack WindDirection into bins
mapping = {
    0: "N",
    1: "NE",
    2: "E",
    3: "SE",
    4: "S",
    5: "SW",
    6: "W",
    7: "NW"
}
def get_wind_direction(degrees):
    cat = round(degrees / 45) % 8
for c, df in data.items():
    df["WindDirection"] = df["WindDirection"].apply(lambda x: round(x * 16 / 360) % 8)

In [116]:
data["Hungaroring"]

,LapTimeZScore,IsPitLap,Compound,RealCompound,TyreLife,FreshTyre,LapNumber,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection
0,1.832314,False,SOFT,C4,2,False,1,18.4,70,985.7,True,26.4,5
1,4.783928,False,SOFT,C4,3,False,2,18.4,70,985.5,True,26.5,4
2,0.463895,False,SOFT,C4,4,False,3,18.4,70,985.5,False,26.7,4
3,-0.051338,False,SOFT,C4,5,False,4,18.4,70,985.5,True,26.5,5
4,-0.205431,False,SOFT,C4,6,False,5,18.6,71,985.5,True,26.5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5352,-0.514558,False,HARD,C3,14,True,44,22.2,56,987.1,False,31.5,3
5353,-0.513614,False,HARD,C3,15,True,45,22.2,55,987.1,False,31.3,5
5354,-0.046417,False,HARD,C3,16,True,46,22.1,55,987.0,False,31.2,4
5355,0.072901,False,HARD,C3,17,True,47,22.1,56,987.1,False,30.9,4


In [91]:
srt = []


print("\n\n")
for c, df in data.items():
    srt.append((c, df.shape[0]))

srt.sort(key=lambda x: x[1], reverse=True)

one = srt[:len(srt) // 2]
two = srt[len(srt) // 2:]

for o, t in zip(one, two):
    c1, r1 = o
    c2, r2 = t
    print(str(c1).ljust(20), str(r1).ljust(20), str(c2).ljust(20), str(r2).ljust(20))

print("\n\n")




Hungaroring          5357                 Melbourne            3151                
Zandvoort            5190                 Suzuka               2754                
Catalunya            5054                 Baku                 2734                
Monte Carlo          4724                 Silverstone          2680                
Sakhir               4414                 Imola                2442                
Montreal             4345                 Miami                2160                
Monza                3898                 Las Vegas            1799                
Mexico City          3843                 Spa-Francorchamps    1592                
Singapore            3778                 Spielberg            1124                
Jeddah               3430                 Paul Ricard          945                 
Yas Marina Circuit   3307                 Austin               918                 





In [75]:
dummies = pd.get_dummies(data["Hungaroring"])

In [110]:
corr = pd.DataFrame()
corr["abs"] = dummies.corr().abs()["LapTimeZScore"]
corr["raw"] = dummies.corr()["LapTimeZScore"]
corr = corr.sort_values(by="abs", ascending=False).drop("LapTimeZScore")

In [111]:
pd.DataFrame(corr["raw"])

,raw
LapNumber,-0.250795
TyreLife,-0.205257
IsPitLap,0.116253
Rainfall,0.091475
RealCompound_C3,-0.067308
Compound_SOFT,0.064216
RealCompound_C4,0.057517
Compound_HARD,-0.055443
RealCompound_C5,0.054173
Compound_MEDIUM,0.018856


In [106]:
normal = 0
dropped = 0
for s in sessions:
    s.laps.columns
    fil = s.laps.loc[:, [
        "Compound",
        "TyreLife",
        "FreshTyre",
        "LapNumber",
    ]]
    normal += fil.shape[0]
    dropped += fil.dropna().shape[0]

In [109]:
dropped

74997